In [ ]:
!pip install h3
import os
import gc
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import h3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.8 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print("\nLoading inventory...")

output_file = "s3_inventory.csv"
df_stations = pd.read_csv(output_file, usecols=['network', 'station', 'lat', 'lon', 'days'])
print(f" -> Found {len(df_stations):,} unique stations globally.")
# Minimum days station must be up
MIN_DAYS = 15
df_stations = df_stations[df_stations['days'] >= MIN_DAYS].reset_index(drop=True)
print(f" -> {len(df_stations):,} stations with >= {MIN_DAYS} days of data.")


Loading inventory...
 -> Found 50,305 unique stations globally.
 -> 35,464 stations with >= 15 days of data.


In [ ]:
df_stations.head()

,network,station,lat,lon,days
0,12,OBS19,9.83424,-104.27936,361
1,12,OBS20,9.84388,-104.28328,360
2,12,OBS21,9.82879,-104.29892,366
3,14,ISR0,67.17750,-50.34310,579
4,16,MG09,47.71915,-69.99715,1605


In [ ]:
# ==========================================
# 2. Grid making and sammpling
# ==========================================
# Define H3 Resolution.
RESOLUTION = 5
print(f"\nSnapping stations to H3 equal-area grid (Resolution {RESOLUTION})...")

try:
    df_stations['hex_id'] = df_stations.apply(lambda row: h3.latlng_to_cell(row['lat'], row['lon'], RESOLUTION), axis=1)
except AttributeError:
    df_stations['hex_id'] = df_stations.apply(lambda row: h3.geo_to_h3(row['lat'], row['lon'], RESOLUTION), axis=1)

print("Filtering for the highest-uptime station per grid cell...")

df_subset = df_stations.sort_values(['hex_id', 'days'], ascending=[True, False]) \
                          .drop_duplicates(subset='hex_id', keep='first') \
                          .reset_index(drop=True)

print(f" -> Downsampled to {len(df_subset):,} stations.")


Snapping stations to H3 equal-area grid (Resolution 5)...
Filtering for the highest-uptime station per grid cell...
 -> Downsampled to 15,088 stations.


In [ ]:
# ==========================================
# 3. Settting distances (60km - 6000km)
# ==========================================
print("\nBuilding spatial index for pairing...")

EARTH_RADIUS_KM = 6371.0

MIN_DIST_KM = 60.0
MAX_DIST_KM = 6000.0

coords_rad = np.radians(df_subset[['lat', 'lon']].values)


tree = BallTree(coords_rad, metric='haversine', leaf_size=40)



Building spatial index for pairing...


In [ ]:

# ---------------------------------------------------------
# 4. Pairing
# ---------------------------------------------------------

# To radians
max_radius_rad = MAX_DIST_KM / EARTH_RADIUS_KM
min_radius_rad = MIN_DIST_KM / EARTH_RADIUS_KM

indices_max = tree.query_radius(coords_rad, r=max_radius_rad)
indices_min = tree.query_radius(coords_rad, r=min_radius_rad)

paired_data = []

for i in range(len(df_subset)):

    valid_neighbors = np.setdiff1d(indices_max[i], indices_min[i])

    unique_valid_neighbors = valid_neighbors[valid_neighbors > i]

    sta1 = df_subset.iloc[i]

    for j in unique_valid_neighbors:
        sta2 = df_subset.iloc[j]


        lat1, lon1 = coords_rad[i]
        lat2, lon2 = coords_rad[j]

        dlon = lon2 - lon1
        dlat = lat2 - lat1
        a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
        c = 2 * np.arcsin(np.sqrt(a))
        distance_km = EARTH_RADIUS_KM * c

        paired_data.append({
            'net1': sta1['network'],
            'sta1': sta1['station'],
            'lat1': sta1['lat'],
            'lon1': sta1['lon'],
            'days1': sta1['days'],
            'net2': sta2['network'],
            'sta2': sta2['station'],
            'lat2': sta2['lat'],
            'lon2': sta2['lon'],
            'days2': sta2['days'],
            'distance_km': round(distance_km, 2)
        })

df_pairs = pd.DataFrame(paired_data)

print(f" -> Success! Found {len(df_pairs):,} valid station pairs.")



 -> Success! Found 40,559,501 valid station pairs.


In [ ]:
!unzip keys_partitioned_year.zip

Archive:  keys_partitioned_year.zip
   creating: year=1969/
   creating: year=1970/
   creating: year=1971/
   creating: year=1972/
   creating: year=1973/
   creating: year=1974/
   creating: year=1975/
   creating: year=1976/
   creating: year=1977/
   creating: year=1978/
   creating: year=1979/
   creating: year=1980/
   creating: year=1981/
   creating: year=1982/
   creating: year=1983/
   creating: year=1984/
   creating: year=1985/
   creating: year=1986/
   creating: year=1987/
   creating: year=1988/
   creating: year=1989/
   creating: year=1990/
   creating: year=1991/
   creating: year=1992/
   creating: year=1993/
   creating: year=1994/
   creating: year=1995/
   creating: year=1996/
   creating: year=1997/
   creating: year=1998/
   creating: year=1999/
   creating: year=2000/
   creating: year=2001/
   creating: year=2002/
   creating: year=2003/
   creating: year=2004/
   creating: year=2005/
   creating: year=2006/
   creating: year=2007/
   creating: year=2008/
   c

In [ ]:
# ---------------------------------------------------------
# 5. Data overlap filtering
# ---------------------------------------------------------

import pyarrow.parquet as pq
from tqdm.notebook import tqdm

PARQUET_DIR = "keys_partitioned_year"
MIN_OVERLAP_DAYS = 15

print("Loading Parquet key index...")
df_idx = pq.read_table(PARQUET_DIR, columns=["network", "station", "year", "yearday"]).to_pandas()
print(f" -> Loaded {len(df_idx):,} records.")

df_idx["day_key"] = list(zip(df_idx["year"].astype(int), df_idx["yearday"].astype(int)))
station_day_series = df_idx.groupby(["network", "station"], observed=True)["day_key"].apply(frozenset)
del df_idx
gc.collect()

print(f" -> {len(station_day_series):,} unique stations indexed.")
print(f"Computing temporal overlap for {len(df_pairs):,} pairs...")

sta1_keys = list(zip(df_pairs["net1"], df_pairs["sta1"]))
sta2_keys = list(zip(df_pairs["net2"], df_pairs["sta2"]))

empty = frozenset()
days1 = [station_day_series.get(k, empty) for k in tqdm(sta1_keys, desc="Lookup sta1")]
days2 = [station_day_series.get(k, empty) for k in tqdm(sta2_keys, desc="Lookup sta2")]

df_pairs["overlap_days"] = [
    len(a & b)
    for a, b in tqdm(zip(days1, days2), total=len(df_pairs), desc="Intersecting")
]

df_filtered = df_pairs[df_pairs["overlap_days"] >= MIN_OVERLAP_DAYS].reset_index(drop=True)

pct_kept = 100 * len(df_filtered) / len(df_pairs) if len(df_pairs) > 0 else 0
print(f" -> {len(df_filtered):,} pairs with >= {MIN_OVERLAP_DAYS} overlapping days")
print(f"    ({pct_kept:.1f}% of {len(df_pairs):,} spatial candidates retained)")

Loading Parquet key index...
 -> Loaded 31,964,420 records.
 -> 53,228 unique stations indexed.
Computing temporal overlap for 40,559,501 pairs...


Lookup sta1:   0%|          | 0/40559501 [00:00<?, ?it/s]

Lookup sta2:   0%|          | 0/40559501 [00:00<?, ?it/s]

Intersecting:   0%|          | 0/40559501 [00:00<?, ?it/s]

 -> 9,447,447 pairs with >= 15 overlapping days
    (23.3% of 40,559,501 spatial candidates retained)


In [ ]:
out_file = "global_station_pairs_R5_15088.csv"
df_filtered.to_csv(out_file, index=False)
print(f"Saved: {out_file}")

Saved: global_station_pairs_R5_15088.csv


In [ ]:
#768683
#3,877,878
#9,447,447
#